# Деревья принятия решений


## Теоретические сведения

**Дерево принятия решений** — алгоритм машинного обучения, который строит правила вида «если–то» для классификации или регрессии. В каждом внутреннем узле дерева проверяется условие вида $x_j \leq t$, где $j$ — индекс признака, $t$ — пороговое значение.

**Критерий Джини** (индекс Джини) вычисляется по формуле:
$$H(X) = 1 - \sum_{k=1}^{K} p_k^2$$
где $p_k$ — доля объектов класса $k$ в выборке $X$, $K$ — количество классов. Минимум (0) достигается, когда все объекты принадлежат одному классу.

**Прирост информации** (information gain) — функционал качества разбиения:
$$Q(X_m, j, t) = H(X_m) - \frac{|X_l|}{|X_m|}H(X_l) - \frac{|X_r|}{|X_m|}H(X_r)$$
где $X_l$ и $X_r$ — левое и правое поддерево после разбиения. Задача состоит в **максимизации** прироста информации.

**Критерии останова** предотвращают переобучение:
- `max_depth` — ограничение максимальной глубины дерева
- `min_samples_leaf` — минимальное число объектов в листе

## 1. Импорт библиотек

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

print("Библиотеки успешно импортированы.")

Библиотеки успешно импортированы.


## 2. Реализация классов узла и листа дерева

Дерево состоит из двух типов узлов:
- **Node** (внутренний узел) — содержит условие разбиения: индекс признака и порог.
- **Leaf** (лист / терминальный узел) — содержит финальный прогноз класса.

In [ ]:
class Node:
  def __init__(self, index, t, true_branch, false_branch):
      self.index = index
      self.t = t
      self.true_branch = true_branch
      self.false_branch = false_branch

class Leaf:
  def __init__(self, data, labels):
      self.data = data
      self.labels = labels
      self.prediction = self.predict()

  def predict(self):
      classes = {}
      for label in self.labels:
          if label not in classes:
            classes[label] = 0
          classes[label] += 1
      prediction = max(classes, key=classes.get)
      return prediction

print("Классы Node и Leaf определены.")

Классы Node и Leaf определены.


## 3. Функция расчёта критерия Джини

**Индекс Джини** оценивает «загрязнённость» узла. Чем меньше его значение, тем чище разбиение:

$$H(X) = 1 - \sum_{k=1}^{K} p_k^2, \quad \text{где } p_k = \frac{1}{|X|}\sum_{i \in X}[y_i = k]$$

In [ ]:
def gini(labels):
  if len(labels) == 0:
    return 0
  classes, counts = np.unique(labels, return_counts=True)
  probabilities = counts / len(labels)
  gini_value = 1 - np.sum(probabilities ** 2)
  return gini_value

## 4. Функция расчёта прироста информации

**Прирост информации** показывает, насколько выбранное разбиение уменьшает неопределённость:

$$Q(X_m, j, t) = H(X_m) - \frac{|X_l|}{|X_m|}H(X_l) - \frac{|X_r|}{|X_m|}H(X_r)$$

Чем больше прирост информации, тем лучше разбиение. Наилучший вопрос выбирается перебором всех признаков и возможных порогов.

In [ ]:
def gain(left_labels, right_labels, root_gini):
  n = len(left_labels) + len(right_labels)
  if n == 0:
    return 0
  p_left = len(left_labels)/n
  p_right = len(right_labels)/n
  information_gain = root_gini - p_left * gini(left_labels) - p_right * gini(right_labels)
  return information_gain

## 5. Вспомогательная функция разбиения узла

In [ ]:
def split(data, labels, column_index, t):
  left = np.where(data[:, column_index] <= t)
  right = np.where(data[:, column_index] >  t)

  true_data = data[left]
  false_data = data[right]
  true_labels = labels[left]
  false_labels = labels[right]

  return true_data, false_data, true_labels, false_labels

print("Функция split определена.")

Функция split определена.


## 6. Критерии останова

Без ограничений дерево разрастается до тех пор, пока в каждом листе не окажется ровно по одному объекту. Это ведёт к **переобучению**: MSE на обучающей выборке равен нулю, но модель плохо обобщается на новые данные.

Реализуем **два критерия останова**:
1. **`max_depth`** — ограничение максимальной глубины дерева (уровней с вопросами).
2. **`min_samples_leaf`** — минимальное число объектов в листе: не разбиваем узел, если в любой из ветвей окажется меньше `min_samples_leaf` объектов.

Функция поиска наилучшего разбиения также учитывает оба критерия.

In [ ]:
# Нахождение наилучшего разбиения с критериями останова
MAX_DEPTH = 5
MIN_SAMPLES_LEAF = 3

def find_best_split(data, labels):
  root_gini = gini(labels)
  best_gain = 0
  best_t = None
  best_index = None
  n_features = data.shape[1]

  for index in range(n_features):
      unique_values = np.unique(data[:, index])
      t_values = (unique_values[:-1] + unique_values[1:])/2
      for t in t_values:
          true_data, false_data, true_labels, false_labels = split(data, labels, index, t)
          if len(true_labels) < MIN_SAMPLES_LEAF or len(false_labels) < MIN_SAMPLES_LEAF:
              continue
          current_gain = gain(true_labels, false_labels, root_gini)
          if current_gain > best_gain:
              best_gain = current_gain
              best_t = t
              best_index = index

  return best_gain, best_t, best_index

print(f"Критерии останова:")
print(f"1) max_depth = {MAX_DEPTH} (ограничение глубины дерева)")
print(f"2) min_samples_leaf = {MIN_SAMPLES_LEAF} (мин. объектов в листе)")

Критерии останова:
1) max_depth = 5 (ограничение глубины дерева)
2) min_samples_leaf = 3 (мин. объектов в листе)


## 7. Построение дерева

Дерево строится рекурсивно сверху вниз — от корня к листьям. На каждом шаге:
1. Проверяются критерии останова.
2. Если рост не прекращён — находится наилучшее разбиение.
3. Данные делятся на две подвыборки, и функция вызывается рекурсивно для каждой.

In [ ]:
def build_tree(data, labels, depth=0):
  if depth >= MAX_DEPTH:
      return Leaf(data, labels)

  ig, t, index = find_best_split(data, labels)
  if ig == 0:
      return Leaf(data, labels)

  true_data, false_data, true_labels, false_labels = split(data, labels, index, t)

  true_branch  = build_tree(true_data,  true_labels,  depth + 1)
  false_branch = build_tree(false_data, false_labels, depth + 1)

  return Node(index, t, true_branch, false_branch)

print("Функция build_tree определена.")

Функция build_tree определена.


## 8. Функции классификации и вывода дерева

In [ ]:
def classify_object(obj, node):
  if isinstance(node, Leaf):
      return node.prediction
  if obj[node.index] <= node.t:
      return classify_object(obj, node.true_branch)
  else:
      return classify_object(obj, node.false_branch)

def predict(data, tree):
  return [classify_object(obj, tree) for obj in data]

def print_tree(node, spacing="", feature_names=None):
  if isinstance(node, Leaf):
      print(spacing + "Прогноз:", node.prediction)
      return
  feat = f"признак_{node.index}" if feature_names is None else feature_names[node.index]
  print(spacing + f'{feat} <= {node.t:.4f}')
  print(spacing + '--> True:')
  print_tree(node.true_branch,  spacing + "  ", feature_names)
  print(spacing + '--> False:')
  print_tree(node.false_branch, spacing + "  ", feature_names)

print("Вспомогательные функции определены.")

Вспомогательные функции определены.


## 9. Функция подсчёта метрики качества

Используем **точность классификации** (accuracy) — долю правильно предсказанных объектов:

$$\text{Accuracy} = \frac{\text{число правильных предсказаний}}{\text{общее число объектов}}$$

In [ ]:
def accuracy_metric(actual, predicted):
  correct = sum(a == p for a, p in zip(actual, predicted))
  return correct / len(actual)

## 10. Проверка на реальных данных

### 10.1 Генерация датасета

Генерируем датасет для задачи бинарной классификации с помощью `make_classification` из sklearn.

In [ ]:
X, y = make_classification(n_samples=300, n_features=4, n_informative=3, n_redundant=1, n_classes=2, random_state=42)

print(f"Размер датасета: {X.shape}")
print(f"Распределение классов: {np.bincount(y)}")
print(f"Признаки (первые 5 строк):\n{X[:5].round(3)}")
print(f"Метки (первые 5):  {y[:5]}")

Размер датасета: (300, 4)
Распределение классов: [150 150]
Признаки (первые 5 строк):
[[ 0.976  1.479  1.247  1.281]
 [-0.166 -2.785 -2.021 -1.415]
 [ 0.457 -0.804  0.016  1.257]
 [ 0.533 -0.468  0.147  1.125]
 [ 0.537  0.102  0.287  0.666]]
Метки (первые 5):  [1 0 1 0 1]


### 10.2 Разбиение на обучающую и тестовую выборки

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Обучающая выборка: {X_train.shape[0]} объектов")
print(f"Тестовая выборка:  {X_test.shape[0]} объектов")

Обучающая выборка: 210 объектов
Тестовая выборка:  90 объектов


### 10.3 Обучение самописного дерева решений

In [ ]:
my_tree = build_tree(X_train, y_train)

train_pred_custom = predict(X_train, my_tree)
test_pred_custom  = predict(X_test,  my_tree)

train_acc_custom = accuracy_metric(y_train, train_pred_custom)
test_acc_custom  = accuracy_metric(y_test,  test_pred_custom)

print("Самописное дерево решений")
print(f"Точность на обучении: {train_acc_custom:.4f}")
print(f"Точность на тесте: {test_acc_custom:.4f}")

Самописное дерево решений
Точность на обучении: 0.9381
Точность на тесте: 0.8333


### 10.4 Структура построенного дерева

In [ ]:
feature_names = [f"x{i}" for i in range(X.shape[1])]

print("Структура самописного дерева:")
print("=" * 50)
print_tree(my_tree, feature_names=feature_names)

Структура самописного дерева:
x0 <= -0.0004
--> True:
  x1 <= -0.4715
  --> True:
    x1 <= -1.7811
    --> True:
      x1 <= -2.0972
      --> True:
        x0 <= -0.3341
        --> True:
          Прогноз: 0
        --> False:
          Прогноз: 0
      --> False:
        Прогноз: 1
    --> False:
      x0 <= -0.2229
      --> True:
        Прогноз: 0
      --> False:
        Прогноз: 0
  --> False:
    x0 <= -1.0839
    --> True:
      x2 <= 0.4400
      --> True:
        x3 <= 0.2179
        --> True:
          Прогноз: 0
        --> False:
          Прогноз: 0
      --> False:
        Прогноз: 1
    --> False:
      Прогноз: 1
--> False:
  x3 <= 0.8963
  --> True:
    x1 <= -0.0246
    --> True:
      x1 <= -0.7603
      --> True:
        x0 <= 0.9052
        --> True:
          Прогноз: 0
        --> False:
          Прогноз: 1
      --> False:
        x0 <= 0.2387
        --> True:
          Прогноз: 0
        --> False:
          Прогноз: 0
    --> False:
      Прогноз: 1
  --

### 10.5 Обучение дерева из sklearn для сравнения

Обучим `DecisionTreeClassifier` из sklearn с теми же параметрами ограничения, чтобы сравнить результаты.

In [ ]:
sklearn_tree = DecisionTreeClassifier(criterion='gini', max_depth=MAX_DEPTH, min_samples_leaf=MIN_SAMPLES_LEAF, random_state=42)
sklearn_tree.fit(X_train, y_train)

train_pred_sklearn = sklearn_tree.predict(X_train)
test_pred_sklearn  = sklearn_tree.predict(X_test)

train_acc_sklearn = accuracy_score(y_train, train_pred_sklearn)
test_acc_sklearn  = accuracy_score(y_test,  test_pred_sklearn)

print("Дерево решений sklearn")
print(f"Точность на обучении: {train_acc_sklearn:.4f}")
print(f"Точность на тесте: {test_acc_sklearn:.4f}")

Дерево решений sklearn
Точность на обучении: 0.9429
Точность на тесте: 0.7778


### 10.6 Сравнение предсказаний объект за объектом

Проверим, что предсказания самописного дерева и sklearn совпадают на тестовой выборке.

In [ ]:
test_pred_custom_arr  = np.array(test_pred_custom)
test_pred_sklearn_arr = np.array(test_pred_sklearn)

matches = (test_pred_custom_arr == test_pred_sklearn_arr)
n_match = matches.sum()
n_total = len(matches)

print(f"Совпадений предсказаний: {n_match} из {n_total}")

if n_match == n_total:
  print("\nРезультаты самописного дерева и sklearn ПОЛНОСТЬЮ СОВПАДАЮТ!")
else:
  print(f"\nРасхождения в {n_total - n_match} объектах.")
  diff_idx = np.where(~matches)[0]
  print(f"Индексы расхождений: {diff_idx}")
  print("Самописное:", test_pred_custom_arr[diff_idx])
  print("sklearn:", test_pred_sklearn_arr[diff_idx])

Совпадений предсказаний: 83 из 90

Расхождения в 7 объектах.
Индексы расхождений: [17 23 30 41 78 80 87]
Самописное: [0 0 0 1 0 0 0]
sklearn: [1 1 1 0 1 1 1]


### 10.7 Итоговая таблица сравнения

In [ ]:
print("=" * 55)
print(f"{'Модель':<25} {'Точность (обучение)':>20} {'Точность (тест)':>15}")
print("-" * 55)
print(f"{'Самописное дерево':<25} {train_acc_custom:>20.4f} {test_acc_custom:>15.4f}")
print(f"{'DecisionTree sklearn':<25} {train_acc_sklearn:>20.4f} {test_acc_sklearn:>15.4f}")
print("=" * 55)

print(f"\nПараметры дерева:")
print(f"Критерий информативности: Джини")
print(f"max_depth: {MAX_DEPTH}")
print(f"min_samples_leaf: {MIN_SAMPLES_LEAF}")

Модель                     Точность (обучение) Точность (тест)
-------------------------------------------------------
Самописное дерево                       0.9381          0.8333
DecisionTree sklearn                    0.9429          0.7778

Параметры дерева:
Критерий информативности: Джини
max_depth: 5
min_samples_leaf: 3


## 11. Демонстрация влияния критериев останова

Покажем, как изменение параметров останова влияет на переобучение. Обучим дерево без ограничений (как модель с переобучением) и сравним с ограниченным.

In [ ]:
print("Влияние критериев останова на точность модели:")
print("=" * 70)
print(f"{'Шаг':<5} {'max_depth':>10} {'min_leaf':>10} {'Train Acc':>12} {'Test Acc':>12} {'Разница':>10}")
print("-" * 70)

configs = [
  (None, 1, "Без ограничений (переобучение)"),
  (10, 1, "max_depth=10"),
  (5, 1, "max_depth=5"),
  (5, 3, "max_depth=5, min_leaf=3"),
    (3, 5, "max_depth=3, min_leaf=5"),
]

for step, (md, ml, label) in enumerate(configs):
  clf = DecisionTreeClassifier(criterion='gini', max_depth=md, min_samples_leaf=ml, random_state=42)
  clf.fit(X_train, y_train)
  tr_acc = accuracy_score(y_train, clf.predict(X_train))
  te_acc = accuracy_score(y_test, clf.predict(X_test))
  md_str = str(md) if md is not None else "None"
  print(f"{step:<5} {md_str:>10} {ml:>10} {tr_acc:>12.4f} {te_acc:>12.4f} {tr_acc - te_acc:>10.4f}")

print("-" * 70)
print("Разница = Train Acc − Test Acc (чем меньше — тем меньше переобучение)")

Влияние критериев останова на точность модели:
Шаг    max_depth   min_leaf    Train Acc     Test Acc    Разница
----------------------------------------------------------------------
0           None          1       1.0000       0.8333     0.1667
1             10          1       1.0000       0.8333     0.1667
2              5          1       0.9619       0.8444     0.1175
3              5          3       0.9429       0.7778     0.1651
4              3          5       0.8810       0.7778     0.1032
----------------------------------------------------------------------
Разница = Train Acc − Test Acc (чем меньше — тем меньше переобучение)



# 12. Выводы


1. **Критерий Джини** реализован корректно: для чистого узла (все объекты одного класса) возвращает 0; для равномерно смешанного — максимальное значение (0.5 для двух классов).

2. **Прирост информации** реализован по формуле $Q = H(X_m) - \frac{|X_l|}{|X_m|}H(X_l) - \frac{|X_r|}{|X_m|}H(X_r)$. Функция `find_best_split` перебирает все признаки и пороги, выбирая разбиение с максимальным значением.

3. **Критерии останова** реализованы два:
   - `max_depth` — ограничение глубины дерева предотвращает рост в сложное ветвистое дерево;
   - `min_samples_leaf` — минимальное число объектов в листе не допускает создания узлов, запоминающих шум.
   Эксперименты показали: без ограничений дерево идеально запоминает обучающую выборку (точность = 1.0), но заметно хуже обобщается на тесте. Применение критериев останова снизило разницу между качеством на обучении и тесте.

4. **Метрика качества** (точность) реализована как доля правильно классифицированных объектов.